In [47]:
 
from azure.identity import InteractiveBrowserCredential

cred = InteractiveBrowserCredential(tenant_id="8aefdf9f-8780-46bf-8fb7-4c924653a8be")

token = cred.get_token("https://storage.azure.com/.default")

print(token)

 

INFO Request URL: 'https://login.microsoftonline.com/8aefdf9f-8780-46bf-8fb7-4c924653a8be/v2.0/.well-known/openid-configuration'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.1 Python/3.11.5 (Windows-10-10.0.22631-SP0)'
No body was attached to the request
INFO Response status: 200
Response headers:
    'Cache-Control': 'max-age=86400, private'
    'Content-Type': 'application/json; charset=utf-8'
    'Strict-Transport-Security': 'REDACTED'
    'X-Content-Type-Options': 'REDACTED'
    'Access-Control-Allow-Origin': 'REDACTED'
    'Access-Control-Allow-Methods': 'REDACTED'
    'P3P': 'REDACTED'
    'x-ms-request-id': '3a8d4e7a-f8da-411d-9dcd-1a0065000800'
    'x-ms-ests-server': 'REDACTED'
    'x-ms-srs': 'REDACTED'
    'Content-Security-Policy-Report-Only': 'REDACTED'
    'X-XSS-Protection': 'REDACTED'
    'Set-Cookie': 'REDACTED'
    'Date': 'Wed, 19 Nov 2025 16:19:40 GMT'
    'Content-Length': '1996'
INFO Request URL: 'https://login.microsoftonli

AccessToken(token='eyJ0eXAiOiJKV1QiLCJhbGciOiJSUzI1NiIsIng1dCI6InJ0c0ZULWItN0x1WTdEVlllU05LY0lKN1ZuYyIsImtpZCI6InJ0c0ZULWItN0x1WTdEVlllU05LY0lKN1ZuYyJ9.eyJhdWQiOiJodHRwczovL3N0b3JhZ2UuYXp1cmUuY29tIiwiaXNzIjoiaHR0cHM6Ly9zdHMud2luZG93cy5uZXQvOGFlZmRmOWYtODc4MC00NmJmLThmYjctNGM5MjQ2NTNhOGJlLyIsImlhdCI6MTc2MzU2ODg4NSwibmJmIjoxNzYzNTY4ODg1LCJleHAiOjE3NjM1NzM4MTIsImFjciI6IjEiLCJhaW8iOiJBVVFBdS84YUFBQUFIRFhqeDFVUElSUFpvb2UyUDBIbHJaVitlbWZzZGhlYkVtaHdUaEJseGt0UWFyOE9haGYwUTAzNG51VDFreVp5M2tqUnJtck9QNXQ0NXNNWGJBQzcwdz09IiwiYW1yIjpbInB3ZCIsInJzYSJdLCJhcHBpZCI6IjA0YjA3Nzk1LThkZGItNDYxYS1iYmVlLTAyZjllMWJmN2I0NiIsImFwcGlkYWNyIjoiMCIsImRldmljZWlkIjoiMjE1OWE4NGMtYjIyZC00NGMzLTgwNzktMDhlMGM2MjdlZWM4IiwiZmFtaWx5X25hbWUiOiJLaWFuaSIsImdpdmVuX25hbWUiOiJBbWlyaG9zc2VpbiIsImdyb3VwcyI6WyJiYzY1MzIwMC1jZjRlLTRkZTUtOGQwMS03NTEyNDIxOTMxMGMiLCI2MjNlMmUwYS1kMDE1LTQyYmMtODk5Yi1hNzYxNmM3OTQ0MjQiLCJjZjYxN2ExMC1lM2M0LTRlOTctOThlNi02NjZiMWYwMTkyOTEiLCJlM2ZmNGUxMS0xZmM0LTQxYTctYmNmOC1kMTJiMDAwYzRhZmYiLCIyOTdlMjUxMi1jMGQ4

# Fusing the detection vehicles info

In [1]:
from pathlib import Path
import sys
import os
import django
from django.conf import settings


project_root = Path(r".\backend")
sys.path.insert(0, str(project_root))

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "backend.processor.settings")

django.setup()

print("Loaded settings module:", os.environ.get("DJANGO_SETTINGS_MODULE"))
print("DATABASES:", settings.DATABASES)


Loaded settings module: backend.processor.settings
DATABASES: {'default': {'ENGINE': 'django.db.backends.sqlite3', 'NAME': WindowsPath('C:/MyApps/VisionSaver/backend/db.sqlite3'), 'TEST': {'MIRROR': 'default', 'CHARSET': None, 'COLLATION': None, 'MIGRATE': True, 'NAME': None}, 'ATOMIC_REQUESTS': False, 'AUTOCOMMIT': True, 'CONN_MAX_AGE': 0, 'CONN_HEALTH_CHECKS': False, 'OPTIONS': {}, 'TIME_ZONE': None, 'USER': '', 'PASSWORD': '', 'HOST': '', 'PORT': ''}}


In [50]:
from record.models import Record, RecordLog # type: ignore
from ai.models import AutoDetection, DetectionLines # type: ignore
from asgiref.sync import sync_to_async

detections = await sync_to_async(list)(AutoDetection.objects.all())


In [131]:
import pandas as pd
movement_to_keys = {"through": 10, "left": 20, "right": 30}


def get_auto_detection_movements(auto_df, lines, record_id):
    min_max = auto_df.groupby("track_id")["time"].agg({"min", "max"})
    min_max["diff"] = min_max["max"] - min_max["min"]
    zone_numbers = {key:len(value) for key, value in lines.items()}
    significant_tracks = min_max[min_max["diff"] > 1].index.tolist()
    significant_auto_df = auto_df[auto_df["track_id"].isin(significant_tracks)]
    automobile_cls_ids = [2, 5, 7]
    significant_auto_df_vehicles = significant_auto_df[significant_auto_df["cls_id"].isin(automobile_cls_ids)]
    significant_auto_df_vehicles["line_index_key"] = significant_auto_df_vehicles["line_index"].apply(
        lambda x: next((movement_to_keys[k] for k in movement_to_keys if k in str(x).lower()), None)
    )
    significant_auto_df_vehicles = significant_auto_df_vehicles.dropna(subset=["line_index_key"])
    significant_auto_df_vehicles["final_turn_movement"] = None
    groups = significant_auto_df_vehicles.groupby("track_id")[["time", "line_index_key", "zone_index"]]
    for track_id, group in groups:
        subgroups = {
            line_key: list(int(z) for z in sub["zone_index"].unique())
            for line_key, sub in group.groupby("line_index_key")
        }
        zone_numbers_mapped = {}
        for line_name, zones_list in lines.items():
            mapped_key = next(
            (movement_to_keys[k] for k in movement_to_keys if k in str(line_name).lower()),
            None,
            )
            if mapped_key is not None:
                zone_numbers_mapped[mapped_key] = len(zones_list)
        
        # replace the string-keyed zone_numbers with numeric-keyed mapping so subsequent checks work
        zone_numbers = zone_numbers_mapped
        for line_key, zones in subgroups.items():
            if len(zones) == zone_numbers.get(line_key, 0):
                # check if zones is incremental, if not we are not interested
                
                if zones == sorted(zones):
                    
                    significant_auto_df_vehicles.loc[significant_auto_df_vehicles["track_id"] == track_id, "final_turn_movement"] = line_key
                    # TODO: we might need to check this value against other line keys. This
                    # might not be the end of the process. Check this later
                    break
    
    significant_auto_df_vehicles.dropna(subset=["final_turn_movement"], inplace=True)
    significant_auto_df_vehicles["record_id"] = record_id
    return significant_auto_df_vehicles

def get_matching_data(manaul_df, auto_df, auto_lines, record_id):
    significant_auto_df_vehicles = get_auto_detection_movements(auto_df, auto_lines, record_id)
    significant_auto_df_vehicles["matched"] = False
    manual_df_temp = manaul_df.copy()
    manual_df_temp["line_index_key"] = manual_df_temp["turn_movement"].apply(
        lambda x: movement_to_keys.get(str(x).lower(), None)
    )
    manual_df_temp["matched"] = False
    detection_time_df = significant_auto_df_vehicles.groupby(["track_id", "final_turn_movement"]).agg({"time": "max"}).rename({"time": "detection_time"}, axis=1).reset_index()
    for index, row in detection_time_df.iterrows():
        track_id = row["track_id"]
        final_turn_movement = row["final_turn_movement"]
        detection_time = row["detection_time"]
        time_window_start = detection_time - 5  # 2 seconds before
        time_window_end = detection_time + 5    # 2 seconds after
        mask = (
            (manual_df_temp["line_index_key"] == final_turn_movement) &
            (manual_df_temp["time"] >= time_window_start) &
            (manual_df_temp["time"] <= time_window_end) &
            (~manual_df_temp["matched"])
        )
        matched_indices = manual_df_temp[mask].index
        manual_df_temp.loc[matched_indices, "matched"] = True
        significant_auto_df_vehicles.loc[significant_auto_df_vehicles["track_id"] == track_id, "matched"] = True
    return significant_auto_df_vehicles


In [132]:
from tqdm import tqdm
final_df = pd.DataFrame()
starting_number = 1
for detection in tqdm(detections):
    record_id = detection.record_id
    record = await sync_to_async(list)(Record.objects.filter(id=record_id))
    if record:
        record = record[0]
        if record.finished_detecting:
            record_manual_logs = await sync_to_async(list)(RecordLog.objects.filter(record_id=record_id))
            record_manual_df_dict = {"time":[], "turn_movement":[], "record_id": []}

            for log in record_manual_logs:
                record_manual_df_dict["time"].append(log.time)
                record_manual_df_dict["turn_movement"].append(log.turn_movement)
                record_manual_df_dict["record_id"].append(record_id)
            record_manual_df = pd.DataFrame(record_manual_df_dict)
            detection_auto_df = pd.read_csv(detection.file_name)
            detection_lines = await sync_to_async(list)(DetectionLines.objects.filter(record_id=record_id))
            auto_lines = detection_lines[0].lines
            matching_data = get_matching_data(record_manual_df, detection_auto_df, auto_lines, record_id)
            # Remap track_id
            unique_track_ids = sorted(matching_data["track_id"].unique())
            track_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_track_ids, start=starting_number)}
            matching_data.loc[:, "track_id"] = matching_data["track_id"].map(track_id_map)
            starting_number += len(unique_track_ids) + 1
            final_df = pd.concat([final_df, matching_data], ignore_index=True)

  3%|▎         | 4/115 [00:00<00:14,  7.90it/s]C:\Users\amki003\AppData\Local\Temp\ipykernel_28996\252472832.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  significant_auto_df_vehicles["line_index_key"] = significant_auto_df_vehicles["line_index"].apply(
  9%|▊         | 10/115 [00:01<00:14,  7.40it/s]C:\Users\amki003\AppData\Local\Temp\ipykernel_28996\252472832.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  significant_auto_df_vehicles["line_index_key"] = significant_auto_df_vehicles["line_inde

In [133]:
final_df.to_csv("vehicle_movements.csv", index=False)

In [134]:
final_df

,track_id,x1,y1,x2,y2,cls_id,confidence,time,in_area,line_index,zone_index,line_index_key,final_turn_movement,record_id,matched
0,1,0.435003,0.221162,0.499184,0.322312,2,0.811245,0.066733,True,through,0,10.0,10.0,90,True
1,2,0.512382,0.363224,0.606207,0.552886,2,0.477426,0.066733,True,through,0,10.0,10.0,90,True
2,3,0.556482,0.098199,0.590168,0.145772,2,0.365922,0.066733,True,through,0,10.0,10.0,90,True
3,1,0.434532,0.221814,0.498774,0.323175,2,0.792563,0.100100,True,through,0,10.0,10.0,90,True
4,2,0.512257,0.363086,0.606152,0.552892,2,0.470074,0.100100,True,through,0,10.0,10.0,90,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3140251,10510,0.360873,0.432363,0.479815,0.680138,2,0.748573,893.422000,True,through,0,10.0,10.0,221,True
3140252,10510,0.357074,0.442372,0.479543,0.698436,2,0.813484,893.455000,True,through,0,10.0,10.0,221,True
3140253,10510,0.352698,0.454708,0.478715,0.719037,2,0.769498,893.489000,True,through,0,10.0,10.0,221,True
3140254,10510,0.291467,0.649023,0.453555,1.004230,2,0.476249,893.889000,True,through,1,10.0,10.0,221,True


In [4]:
from django.conf import settings
import os
import cv2
from asgiref.sync import sync_to_async
from record.models import Record # type: ignore
media_root = settings.MEDIA_ROOT
records = await sync_to_async(list)(Record.objects.all())
video_data = {"record_id": [], "width": [], "height": []}
for record in records:
    record_id = record.id
    file_path = os.path.join(media_root, f"{record_id}.mkv")
    if not os.path.exists(file_path):
        file_path = os.path.join(media_root, f"{record_id}.mp4")
        if not os.path.exists(file_path):
            continue
    
    # Get video height and width
    cap = cv2.VideoCapture(file_path)
    if not cap.isOpened():
        print(f"Failed to open video file: {file_path}")
        continue
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    video_data["record_id"].append(record_id)
    video_data["width"].append(width)
    video_data["height"].append(height)

import pandas as pd
video_df = pd.DataFrame(video_data)
video_df.to_csv("video_dimensions.csv", index=False)



# Load videos movements and video dimensions

In [1]:
import pandas as pd
final_df = pd.read_csv("vehicle_movements.csv")
video_df = pd.read_csv("video_dimensions.csv")

In [2]:
final_df

,track_id,x1,y1,x2,y2,cls_id,confidence,time,in_area,line_index,zone_index,line_index_key,final_turn_movement,record_id,matched
0,1,0.435003,0.221162,0.499184,0.322312,2,0.811245,0.066733,True,through,0,10.0,10.0,90,True
1,2,0.512382,0.363224,0.606207,0.552886,2,0.477426,0.066733,True,through,0,10.0,10.0,90,True
2,3,0.556482,0.098199,0.590168,0.145772,2,0.365922,0.066733,True,through,0,10.0,10.0,90,True
3,1,0.434532,0.221814,0.498774,0.323175,2,0.792563,0.100100,True,through,0,10.0,10.0,90,True
4,2,0.512257,0.363086,0.606152,0.552892,2,0.470074,0.100100,True,through,0,10.0,10.0,90,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3140251,10510,0.360873,0.432363,0.479815,0.680138,2,0.748573,893.422000,True,through,0,10.0,10.0,221,True
3140252,10510,0.357074,0.442372,0.479543,0.698436,2,0.813484,893.455000,True,through,0,10.0,10.0,221,True
3140253,10510,0.352698,0.454708,0.478715,0.719037,2,0.769498,893.489000,True,through,0,10.0,10.0,221,True
3140254,10510,0.291467,0.649023,0.453555,1.004230,2,0.476249,893.889000,True,through,1,10.0,10.0,221,True


# Augment data

In [4]:
from tqdm import tqdm
temp_df = final_df.copy()
groups = temp_df.groupby("record_id")
augment_size = 10
max_record_id = temp_df["record_id"].max()
import numpy as np
def rotate_point(x, y, cx, cy, angle_rad):
    cos_angle = np.cos(angle_rad)
    sin_angle = np.sin(angle_rad)
    x_new = cos_angle * (x - cx) - sin_angle * (y - cy) + cx
    y_new = sin_angle * (x - cx) + cos_angle * (y - cy) + cy
    return x_new, y_new
for record_id, group in tqdm(groups, total=len(groups)):
    copied_group = group.copy()
    x1, y1, x2, y2 = copied_group["x1"].values, copied_group["y1"].values, copied_group["x2"].values, copied_group["y2"].values
    for i in range(1, augment_size + 1):
        max_record_id += 1
        # Get random rotation angle between -pi and pi
        angle_rad = np.random.uniform(-np.pi/2, 0)
        # video_info = video_df[video_df["record_id"] == record_id]
        # if video_info.empty:
        #     continue
        new_x_1, new_y_1 = rotate_point(x1, y1, 0, 0, angle_rad)
        new_x_2, new_y_2 = rotate_point(x2, y2, 0, 0, angle_rad)
        augmented_group = copied_group.copy()
        augmented_group["x1"] = new_x_1
        augmented_group["y1"] = new_y_1
        augmented_group["x2"] = new_x_2
        augmented_group["y2"] = new_y_2
        augmented_group["record_id"] = max_record_id
        temp_df = pd.concat([temp_df, augmented_group], ignore_index=True)
temp_df.to_csv("augmented_vehicle_movements.csv", index=False)

100%|██████████| 71/71 [07:43<00:00,  6.52s/it]


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
class TurnMovementClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(3, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.mlp(x)
    
class TurnMovementDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

SyntaxError: incomplete input (2019922348.py, line 19)

In [ ]:
model = TurnMovementClassifier(num_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
df = pd.read_csv("augmented_vehicle_movements.csv")


In [ ]:

turn_movement_map = {10: 0, 20: 1, 30: 2}  # through:0, left:1, right:2
from torch.utils.data import DataLoader
all_record_metrics = {}
EPOCHS_PER_RECORD = 5
for record_id, record_group in df.groupby("record_id"):

    print(f"\n==============================")
    print(f" Training on record {record_id} ")
    print(f"==============================\n")

    features_list = []
    labels_list = []

    for track_id, track_group in record_group.groupby("track_id"):

        track_group = track_group.sort_values("time")

        x_c = ((track_group["x1"] + track_group["x2"]) / 2).values
        y_c = ((track_group["y1"] + track_group["y2"]) / 2).values

        dx = np.diff(x_c, prepend=x_c[0])
        dy = np.diff(y_c, prepend=y_c[0])

        start_heading = np.arctan2(dy[0], dx[0])
        end_heading   = np.arctan2(dy[-1], dx[-1])

        delta = np.arctan2(np.sin(end_heading - start_heading),
                           np.cos(end_heading - start_heading))

        final_x = x_c[-1]
        final_y = y_c[-1]

        feature_vec = torch.tensor([delta, final_x, final_y], dtype=torch.float32)

        label = turn_movement_map[track_group["final_turn_movement"].iloc[0]]

        features_list.append(feature_vec)
        labels_list.append(label)

    # dataset
    dataset = TurnMovementDataset(features_list, labels_list)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    # Train & evaluate metrics
    epoch_losses = []
    epoch_accuracies = []

    for epoch in range(EPOCHS_PER_RECORD):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for batch_features, batch_labels in dataloader:
            batch_labels = batch_labels.long()

            optimizer.zero_grad()
            logits = model(batch_features)
            loss = loss_fn(logits, batch_labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_features.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == batch_labels).sum().item()
            total += batch_labels.size(0)

        avg_loss = total_loss / total
        accuracy = correct / total

        epoch_losses.append(avg_loss)
        epoch_accuracies.append(accuracy)

        print(f"Record {record_id} | Epoch {epoch+1}/{EPOCHS_PER_RECORD} | "
              f"Loss: {avg_loss:.4f} | Accuracy: {accuracy:.3f}")

    # Save metrics
    all_record_metrics[record_id] = {
        "losses": epoch_losses,
        "accuracies": epoch_accuracies
    }

    # SAVE MODEL AFTER EACH RECORD
    torch.save(model.state_dict(), f"turn_movement_model_latest.pth")
    print(f"Model saved after record {record_id}.")


Creating the dataset for record: 19
Training on record 19, epoch 1/5
Training on record 19, epoch 2/5
Training on record 19, epoch 3/5
Training on record 19, epoch 4/5
Training on record 19, epoch 5/5
Creating the dataset for record: 20
Training on record 20, epoch 1/5
Training on record 20, epoch 2/5
Training on record 20, epoch 3/5
Training on record 20, epoch 4/5
Training on record 20, epoch 5/5
Creating the dataset for record: 21
Training on record 21, epoch 1/5
Training on record 21, epoch 2/5
Training on record 21, epoch 3/5
Training on record 21, epoch 4/5
Training on record 21, epoch 5/5
Creating the dataset for record: 22
Training on record 22, epoch 1/5
Training on record 22, epoch 2/5
Training on record 22, epoch 3/5
Training on record 22, epoch 4/5
Training on record 22, epoch 5/5
Creating the dataset for record: 88
Training on record 88, epoch 1/5
Training on record 88, epoch 2/5
Training on record 88, epoch 3/5
Training on record 88, epoch 4/5
Training on record 88, epoch

# Test

In [6]:
import torch

model = TurnMovementClassifier(hidden_size=64, num_classes=3)
model.load_state_dict(torch.load("turn_movement_kian/turn_movement_model_record_931.pth"))
model.eval()

TurnMovementClassifier(
  (lstm): LSTM(1, 64, num_layers=2, batch_first=True, bidirectional=True)
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (xy_mlp): Sequential(
    (0): Linear(in_features=2, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=16, bias=True)
  )
  (final_mlp): Sequential(
    (0): Linear(in_features=144, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=3, bias=True)
  )
)

In [15]:
record_id = 500
test_data = df[df["record_id"] == record_id]
print("Shape:", test_data.shape)
print("Number of vehicles", test_data["track_id"].nunique())

Shape: (187305, 15)
Number of vehicles 557


In [9]:
import pandas as pd
df = pd.read_csv("augmented_vehicle_movements.csv")

In [ ]:
import numpy as np
import torch

def extract_features_from_track(track_group):

    track_group = track_group.sort_values("time")

    x_c = ((track_group["x1"] + track_group["x2"]) / 2).values
    y_c = ((track_group["y1"] + track_group["y2"]) / 2).values

    dx = np.diff(x_c, prepend=x_c[0])
    dy = np.diff(y_c, prepend=y_c[0])

    start_heading = np.arctan2(dy[0], dx[0])
    end_heading   = np.arctan2(dy[-1], dx[-1])

    # stable heading difference
    delta = np.arctan2(
        np.sin(end_heading - start_heading),
        np.cos(end_heading - start_heading)
    )

    final_x = x_c[-1]
    final_y = y_c[-1]

    return torch.tensor([delta, final_x, final_y], dtype=torch.float32)
def predict_turn_movement(model, track_group, turn_movement_map):
    feature_vec = extract_features_from_track(track_group).unsqueeze(0)  # shape (1,3)
    logits = model(feature_vec)
    pred = torch.argmax(logits, dim=1).item()

    # reverse map: 0→10, 1→20, 2→30
    reverse_map = {v:k for k,v in turn_movement_map.items()}
    return reverse_map[pred]

turn_movement_map = {10:0, 20:1, 30:2}  # same used in training
vehicle_prediction = predict_turn_movement(model, some_track_df, turn_movement_map)
print("Predicted movement:", vehicle_prediction)

Sample 0: True Label = 1, Predicted Label = 0
Sample 1: True Label = 0, Predicted Label = 2
Sample 2: True Label = 1, Predicted Label = 1
Sample 3: True Label = 0, Predicted Label = 2
Sample 4: True Label = 0, Predicted Label = 0
Sample 5: True Label = 1, Predicted Label = 1
Sample 6: True Label = 1, Predicted Label = 1
Sample 7: True Label = 0, Predicted Label = 2
Sample 8: True Label = 1, Predicted Label = 1
Sample 9: True Label = 2, Predicted Label = 2
Sample 10: True Label = 2, Predicted Label = 2
Sample 11: True Label = 1, Predicted Label = 0
Sample 12: True Label = 0, Predicted Label = 2
Sample 13: True Label = 0, Predicted Label = 2
Sample 14: True Label = 0, Predicted Label = 2
Sample 15: True Label = 1, Predicted Label = 0
Sample 16: True Label = 0, Predicted Label = 2
Sample 17: True Label = 1, Predicted Label = 0
Sample 18: True Label = 0, Predicted Label = 2
Sample 19: True Label = 0, Predicted Label = 2
Sample 20: True Label = 0, Predicted Label = 2
Sample 21: True Label =